# Ring Resonator Analysis, Part 4: Per-Die FSR Aggregation

Grouping hundreds of per-device results by die and radius would normally require walking a directory tree or maintaining a separate index file that maps filenames to their parameters. Because radius and die were stored as tags at upload time, a single `query_files(...).groupby(...)` call does that work here.

In the previous notebook each device got an individual FSR measurement. Now we aggregate those per-device values to the die level.

The key subtlety here is that rings of different radii have different FSRs (larger radius = longer circumference = smaller FSR), so averaging FSR across all radii on a die would be meaningless. We group by **both die and radius** and compute a mean FSR for each (die, radius) combination separately. That gives one die-level result per ring geometry, which is what the wafer map in notebook 5 will visualise.

The function also produces the die coordinates needed for the wafer map. These are read from the `die` tag of the input files, not from inside the JSON data.

## Setup

In [ ]:
import getpass
import json
import shutil
from pathlib import Path

import gfhub
import matplotlib.pyplot as plt
import numpy as np
from gfhub import nodes
from PIL import Image
from tqdm.auto import tqdm

client = gfhub.Client()
user = getpass.getuser()
print(f"Running as user: {user}")

## Load sample data

We download the FSR JSON files for one die and one ring radius as a sanity check before defining the pipeline function. The `groupby("die", "radius_nm")` call partitions the results by the combination of both tags, working because die and radius were set as structured tags at upload time rather than encoded in filenames.

In [ ]:
entries = client.query_files(
    tags=[user, "project:tutorial_rings", ".json", "die:-1,-1", "radius_nm:5000"]
)
ids = [entry["id"] for entry in entries]
tags = [
    [gfhub.tags.into_string(t) for t in entry["tags"].values()]
    for entry in entries
]

sample_dir = Path("sample_die")
shutil.rmtree(sample_dir, ignore_errors=True)
sample_dir.mkdir()

sample_paths = []
for id_, path in zip(ids, [sample_dir / f"{id_}.json" for id_ in ids]):
    client.download_file(id_, path)
    sample_paths.append(path)

print(f"Downloaded {len(sample_paths)} files")
print("Sample tags:", tags[0])

## Defining the die aggregation function

`fsr_for_radius_within_die` receives all the FSR JSON files for one (die, radius) group. It reads `fsr_mean` from each JSON, converts from µm to nm, and computes the mean and standard deviation across devices. The `radius_nm` tag is read from the first file's tags to set up the output filenames and labels.

The JSON output includes `die_x`, `die_y`, `mean`, and `std`. The `mean` field is what the wafer map will plot.

In [ ]:
def fsr_for_radius_within_die(
    files: list[Path],
    tags: list[list[str]],
    /,
) -> tuple[Path, Path]:
    """Aggregate per-device FSR measurements for one die and one ring radius.

    Converts fsr_mean from µm to nm before averaging.
    """
    # Read the radius from the first file's tags
    radius_nm = int(next(t[10:] for t in tags[0] if t.startswith("radius_nm:")))

    fsr_values = []
    for file in files:
        data = json.loads(file.read_text())
        fsr = data.get("fsr_mean")
        if fsr is not None:
            fsr_values.append(1000 * fsr)  # convert µm to nm

    if not fsr_values:
        msg = f"No valid FSR measurements for radius {radius_nm} nm"
        raise ValueError(msg)

    fsr_values = np.array(fsr_values)
    mean_fsr = float(np.mean(fsr_values))
    std_fsr = float(np.std(fsr_values))

    # Extract die coordinates from the first file's tags
    die_x, die_y = None, None
    for tag in tags[0]:
        if tag.startswith("die:"):
            die_x, die_y = [int(c) for c in tag.split(":", 1)[1].split(",")]
            break

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.bar(range(len(fsr_values)), fsr_values, alpha=0.7)
    ax.axhline(mean_fsr, color="r", linestyle="--", label=f"Mean = {mean_fsr:.4f} nm")
    ax.axhline(mean_fsr + std_fsr, color="orange", linestyle=":", alpha=0.7)
    ax.axhline(mean_fsr - std_fsr, color="orange", linestyle=":", alpha=0.7, label=f"±1σ = {std_fsr:.4f} nm")
    ax.set_xlabel("Device Index")
    ax.set_ylabel("FSR (nm)")
    ax.set_title(f"Die ({die_x}, {die_y}) — R = {radius_nm} nm rings")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    plot_path = files[0].parent / f"die_fsr_r{radius_nm}nm.png"
    plt.savefig(plot_path, bbox_inches="tight", dpi=100)
    plt.close()

    results = {
        "die_x": die_x,
        "die_y": die_y,
        "mean": mean_fsr,
        "std": std_fsr,
        "num_devices": len(fsr_values),
        "radius_nm": radius_nm,
    }
    results_path = files[0].parent / f"die_fsr_r{radius_nm}nm.json"
    results_path.write_text(json.dumps(results, indent=2))

    return plot_path, results_path

In [ ]:
func_def = gfhub.Function(
    func=fsr_for_radius_within_die,
    dependencies={
        "json": "import json",
        "numpy": "import numpy as np",
        "matplotlib": "import matplotlib.pyplot as plt",
    },
)

In [ ]:
result = func_def.eval(sample_paths, tags)
plot_path, json_path = result["output"]
print(json_path.read_text())
Image.open(plot_path)

In [ ]:
client.add_function(func_def)
print("fsr_for_radius_within_die uploaded.")

## Common tags

The outputs for each (die, radius) group should be tagged with the provenance shared by all devices in that group: the wafer, project, die, radius, and username. Tags that vary between devices (cell name, device position, ring length) should not appear on the aggregated output. `find_common_tags` handles this by keeping only tags where every input file agrees on the same value. Extension tags (keys starting with `.`) are always excluded.

This is the same helper used in the cutback, resistance, and spirals series. It is re-uploaded here so this tutorial can be followed independently.

In [ ]:
def find_common_tags(tags: list[list[str]], /) -> list[str]:
    """Return only the tags that are identical across all input files."""
    common: dict[str, set] = {}
    for _tags in tags:
        for t in _tags:
            if ":" in t:
                key, value = t.split(":", 1)
            else:
                key, value = t, ""
            common.setdefault(key, set()).add(value)
    agreed = {k: next(iter(v)) for k, v in common.items() if len(v) == 1}
    return [k if not v else f"{k}:{v}" for k, v in agreed.items() if not k.startswith(".")]


client.add_function(find_common_tags)
print("find_common_tags uploaded.")

## Creating the pipeline

This pipeline uses a manual trigger only. There is no meaningful auto-trigger here because the function needs all devices from a (die, radius) group at once to compute a meaningful average. A single file upload cannot provide that.

The pipeline loads the group of files and their tags, runs `fsr_for_radius_within_die` to get a plot and a JSON, then uses `find_common_tags` to extract the shared provenance. Both outputs are saved with the common tags so the wafer map in notebook 5 can query them by radius and wafer.

In [ ]:
p = gfhub.Pipeline()

p.trigger = nodes.on_manual_trigger()
p.load_file = nodes.load()
p.load_tags = nodes.load_tags()
p += p.trigger >> p.load_file
p += p.trigger >> p.load_tags

p.die_fsr = nodes.function(function="fsr_for_radius_within_die")
p += p.load_file >> p.die_fsr[0]
p += p.load_tags >> p.die_fsr[1]

p.common_tags = nodes.function(function="find_common_tags")
p += p.load_tags >> p.common_tags

p.save_plot = nodes.save()
p.save_json = nodes.save()
p += p.die_fsr[0] >> p.save_plot[0]
p += p.common_tags >> p.save_plot[1]
p += p.die_fsr[1] >> p.save_json[0]
p += p.common_tags >> p.save_json[1]

confirmation = client.add_pipeline(name="die_fsr_aggregation", schema=p)
print(f"Pipeline ready: {client.pipeline_url(confirmation['id'])}")

## Trigger per (die, radius) group

Because we need to group by both die and radius, we use `groupby("die", "radius_nm")`. Each group gets its own pipeline call, so rings of different radii on the same die are processed independently. The groupby works here because both `die` and `radius_nm` were stored as tags at upload time, so no directory walk or index file is needed to reconstruct the grouping.

In [ ]:
analysis_results = client.query_files(
    name="*_fsr_analysis.json",
    tags=["project:tutorial_rings", user],
).groupby("die", "radius_nm")

print(f"Found {len(analysis_results)} (die, radius) groups")

job_ids = []
for group_key, files in tqdm(analysis_results.items()):
    input_ids = [f["id"] for f in files]
    triggered = client.trigger_pipeline("die_fsr_aggregation", input_ids)
    job_ids.extend(triggered["job_ids"])

print(f"Triggered {len(job_ids)} die analysis jobs")

In [ ]:
jobs = client.wait_for_jobs(job_ids)
print(f"All jobs complete. Statuses: {set(j['status'] for j in jobs)}")

## View a sample result

The plot shows the per-device FSR values as a bar chart with the mean and ±1σ lines. For rings on the same die with the same radius, the FSR values should cluster tightly around the mean — the spread comes from measurement noise and the small group-index variation we added in the simulation. A large spread or outliers would indicate a defective device or a problem with the peak-finding step in notebook 3.

In [ ]:
die_plots = client.query_files(name="die_fsr_*.png", tags=["project:tutorial_rings", user])
print(f"Found {len(die_plots)} die analysis plots")

if die_plots:
    img = Image.open(client.download_file(die_plots[0]["id"]))
    display(img.resize((530, 400)))

## What's next?

Each (die, radius) combination now has a JSON with the mean FSR in nm and the die coordinates. The next notebook groups these by wafer and ring radius and builds a spatial wafer map for each radius.